In [1]:
from sqlalchemy import create_engine, MetaData, Table, select, insert
from sqlalchemy.exc import SQLAlchemyError
from dotenv import load_dotenv
load_dotenv()
import os
from tqdm import tqdm
import pandas as pd
import regex as re

import spacy

from nltk.metrics import distance
import scipy.spatial as spatial
import numpy as np
from scipy.cluster.vq import kmeans


In [3]:
nlp = spacy.load("en_core_web_trf")

/Users/tom/Fine-arts-ML/Fine-Arts-Main/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [15]:
# cleanse tags before upload!
tags_list =['art-style', 'flowers', 'nature', 'still-life', 'vase', 'red', 'white', 'orange', 'green', 'bloom', 'garden', 'paintings', 'petals', 'blossom', 'colorful', 'composition']
def lemmatize_list(list):
    lemma_list = []
    for word in list:
        doc = nlp(word)
        lemma_list.append(" ".join([token.lemma_ for token in doc]))
    return lemma_list
df_tags = pd.DataFrame(tags_list, columns=['name'])
df_tags["split_n"] = df_tags.name.apply(lambda x: re.sub('[^A-Za-z0-9]+', ' ', x))
df_tags["split_n"] = df_tags.split_n.str.split()
df_tags["split_n_lem"] = df_tags.split_n.apply(lemmatize_list)
df_tags['lem_combined'] = df_tags['split_n_lem'].apply(lambda x: ' '.join(x))
#df_tags = df_tags.explode('split_n_lem')
tags_lem_list = df_tags['lem_combined'].unique().tolist()
tags_lem_list

['art style',
 'flower',
 'nature',
 'still life',
 'vase',
 'red',
 'white',
 'orange',
 'green',
 'bloom',
 'garden',
 'painting',
 'petal',
 'blossom',
 'colorful',
 'composition']

In [3]:
def create_db_connection():
    DB_HOST = os.getenv("DB_HOST")
    DB_NAME = os.getenv("DB_NAME")
    DB_USER = os.getenv("DB_USER")
    DB_PASSWORD = os.getenv("DB_PASSWORD")
    engine = create_engine('postgresql+pg8000://'+DB_USER+':'+DB_PASSWORD+'@'+DB_HOST+':5432/'+DB_NAME)
    return engine


def get_tags():
    try:
        engine = create_db_connection()
        df_systag = pd.read_sql_table('oc_systemtag', engine)
        df_tagmap = pd.read_sql_table('oc_systemtag_object_mapping', engine)

    except SQLAlchemyError as e:
        print(f"An error occurred: {e}")
    except Exception as e:
        print(f"An unexpected error occurred: {e}")
    return df_systag, df_tagmap

In [15]:
engine = create_db_connection()

In [4]:
df_systag, df_tagmap = get_tags()

In [23]:
df_dense = df_systag


In [ ]:
def lemmatize_list(list):
    lemma_list = []
    for word in list:
        doc = nlp(word)
        lemma_list.append(" ".join([token.lemma_ for token in doc]))
    return lemma_list

df_dense['name'] = df_dense['name'].str.lower()
df_dense["split_n"] = df_dense.name.apply(lambda x: re.sub('[^A-Za-z0-9]+', ' ', x))
df_dense["split_n"] = df_dense.split_n.str.split()
df_dense["split_n_lem"] = df_dense.split_n.apply(lemmatize_list)
df_dense = df_dense.explode('split_n_lem')
#df_dense = df_dense.groupby('split_n_lem')['id'].agg(list).reset_index()
#df_dense['lowest_id'] = df_dense['id'].apply(min)

In [25]:
df_lem = df_dense[['id','split_n_lem']].copy()
df_lem = df_lem.reset_index()
df_lem_group = df_lem.groupby('split_n_lem')['id'].agg(list).reset_index()

In [26]:
dict_lem_groups = {}
dict_lem_groups = df_lem_group.to_dict(orient='records')

In [27]:
# Get all unique IDs from all dict entries
all_ids = [id_ for d in dict_lem_groups for id_ in d['id']]
unique_ids = set(all_ids)

# Find the maximum ID to generate new unique IDs if needed
max_id = max(all_ids) if all_ids else 0

# Assign a unique ID from the set to each dictionary
for d in dict_lem_groups:
    assigned = False
    # Try to find an unused ID from the dictionary's 'id' list
    for id_ in d['id']:
        if id_ in unique_ids:
            d['unique_id'] = int(id_)
            unique_ids.remove(id_)
            assigned = True
            break
    # If no unused ID found, generate a new globally unique ID
    if not assigned:
        max_id += 1
        d['unique_id'] = max_id

# Create DataFrame
df_systag_new = pd.DataFrame(dict_lem_groups)


In [28]:
df_systag_new = pd.DataFrame(dict_lem_groups)
#df_systag_new = df_systag_new.dropna(subset=['unique_id'])
df_systag_new['unique_id'] = df_systag_new['unique_id'].astype(int)

In [29]:
# Create a mapping from old IDs to new unique IDs
id_to_unique = {}
for _, row in df_systag_new.iterrows():
    for old_id in row['id']:
        id_to_unique[old_id] = row['unique_id']

#Create a list to store the new rows
new_rows = []

# Iterate over each row in df_tagmap
for _, row in df_tagmap.iterrows():
    old_tag_id = row['systemtagid']
    if old_tag_id in id_to_unique:
        new_tag_id = id_to_unique[old_tag_id]
        # Add a new row with the new unique ID
        new_row = row.copy()
        new_row['systemtagid_new'] = new_tag_id
        new_rows.append(new_row)
        # If the new unique ID is different from the old one, add another row with the old ID
        if new_tag_id != old_tag_id:
            old_row = row.copy()
            old_row['systemtagid_new'] = old_tag_id
            new_rows.append(old_row)
    else:
        # If the old tag ID is not found in the mapping, keep it as is
        new_row = row.copy()
        new_row['systemtagid_new'] = old_tag_id
        new_rows.append(new_row)

# Create a new DataFrame from the list of new rows
df_tagmap_updated = pd.DataFrame(new_rows)

# Check for any unmapped values
unmapped_count = df_tagmap_updated[df_tagmap_updated['systemtagid_new'].isna()].shape[0]
if unmapped_count > 0:
    print(f"Warning: {unmapped_count} rows could not be mapped.")

# Display the first 20 rows of the updated df_tagmap_updated
df_tagmap_updated = df_tagmap_updated.reset_index(drop=True)
df_tagmap_updated.head(20)


,objectid,objecttype,systemtagid,systemtagid_new
0,19548,files,116,383
1,19548,files,116,116
2,19548,files,117,117
3,19548,files,118,118
4,19548,files,119,161
5,19548,files,119,119
6,19548,files,120,120
7,19548,files,121,121
8,19548,files,122,122
9,19548,files,123,123


In [30]:
df_tagmap_updated = df_tagmap_updated.drop(columns=['systemtagid'])
df_tagmap_updated = df_tagmap_updated.rename(columns={'systemtagid_new': 'systemtagid'})


In [31]:
###### ONLY RUN THIS ONCE OR YOULL LOOSE ID ##########
#Prepare df_systag_new for database insertion
df_systag_new = df_systag_new.drop(columns=['id'])
df_systag_new = df_systag_new.rename(columns={'unique_id': 'id', 'split_n_lem': 'name'})



In [32]:
df_systag_new['visibility'] = 1
df_systag_new['editable'] = 1
df_systag_new['etag'] = ''
df_systag_new['color']= ''
df_systag_new[:20]

,name,id,visibility,editable,etag,color
0,3d,1029,1,1,,
1,abstract,144,1,1,,
2,abstractart,692,1,1,,
3,abstractdesign,698,1,1,,
4,abstractpainting,1470,1,1,,
5,abstractpattern,704,1,1,,
6,abstractshape,1690,1,1,,
7,accent,610,1,1,,
8,acrylic,1061,1,1,,
9,aesthetic,1324,1,1,,


# UPLOAD DATA TO DB

### might be worth not replacing the tables, just empty them and append "new data" ??

In [ ]:
df_systag_new.to_sql('oc_systemtag', con= engine, if_exists='replace', index=False)

ObjectNotExecutableError: Not an executable object: 'ALTER TABLE oc_systemtag OWNER TO oc_tom;'

In [ ]:
df_tagmap_updated.to_sql('oc_systemtag_object_mapping', con= engine, if_exists='replace', index=False)


36203

# Cluster test

In [ ]:
import numpy as np
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score
from sentence_transformers import SentenceTransformer, util
model = SentenceTransformer("all-MiniLM-L6-v2")

In [ ]:
def choose_classifier(X):
    X1 = X / (X**2).sum(axis=-1, keepdims=True)
    vv = []
    cc = np.arange(2, len(X))
    for nclusters in cc:
        km_model = KMeans(nclusters).fit(X1)
        labels = km_model.labels_
        v = silhouette_score(X1, labels)
        vv.append(v)
    nclusters = cc[np.argmax(vv)]
    return KMeans(nclusters).fit(X1)

In [ ]:
embeddings = model.encode(result2.lemma, show_progress_bar=True, convert_to_numpy=True)

#classifier = choose_classifier(embeddings)


In [ ]:
for i, (v, s) in enumerate(zip(embeddings, result2.lemma)):
    print(classifier.predict(v[np.newaxis]), s)

In [ ]:
from kmeans_pytorch import kmeans
import torch
def detect_clusters(X, nclusters, tol=1e-6):
  X = torch.as_tensor(X)
  assert X.ndim == 2
  # Project the points in a hypersphere
  X1 = X / torch.sqrt(torch.sum(X**2, axis=-1, keepdims=True))

  # Run kmeans on the normalized points with euclidean distance
  cluster_ID, C = kmeans(X1, nclusters, distance='euclidean', tol=tol)
  return cluster_ID, C

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import torch;
#### THE RESUTLS ####
cluster_ID, C = detect_clusters(embeddings, 10)
# Avoid distortion of the angles
plt.axes().set_aspect('equal')
# Initial points
plt.plot(embeddings[:,0], embeddings[:,1], '.')
# Reference circle
theta = torch.linspace(0, 2*np.pi, 1000)
plt.plot(torch.cos(theta), torch.sin(theta), '--k')
plt.plot(embeddings[:,0],embeddings[:,1], '.')
xlim = plt.xlim()
ylim = plt.ylim()
plt.xlim(xlim)
plt.ylim(ylim)

# Draw lines in the directions given by the centroids
R = 20
for c in C:
    plt.plot([0, c[0]*R], [0, c[1]*R]);

plt.grid();

In [ ]:
from sentence_transformers import SentenceTransformer
from sklearn.cluster import KMeans
from sklearn.metrics.pairwise import cosine_similarity

In [ ]:
# Generate embeddings
model = SentenceTransformer('all-MiniLM-L6-v2')
embeddings = model.encode(result2.lemma, show_progress_bar=True, convert_to_numpy=True)

# Cluster using K-Means
kmeans = KMeans(n_clusters=5, random_state=42)
clusters = kmeans.fit_predict(embeddings)

# Print clusters
for tag, cluster in zip(result2.lemma, clusters):
    print(f"{tag}: Cluster {cluster}")